In [14]:
import asyncio
import json
import pandas as pd

from openai import AsyncOpenAI
from tqdm.asyncio import tqdm
import os

In [15]:
data = pd.read_csv("../data/raw/rusentiment_preselected_posts.csv")

In [16]:
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [17]:
data.head()

,label,text
0,neutral,Прорвём информационную блокаду изнутри.
1,negative,"Никогда у меня не будет ""одного приложения для..."
2,skip,"Кури-и тебя не укусит злая собака, потому что ..."
3,neutral,"Есть 3 типа людей:\nУмные, которые делают все ..."
4,neutral,мегафон чет накрыло


In [18]:
TOPICS = [
    "short_reaction",
    "daily_life",
    "relationships",
    "humor",
    "discussion",
    "creative",
    "toxicity",
    "storytelling",
    "philosophical",
    "question",
    "other"
]

INTENTS = [
    "reaction",
    "storytelling",
    "reflection",
    "complaint",
    "discussion",
    "confession",
    "joke",
    "rant",
    "observation",
    "asking"
]

EMOTIONS = [
    "positive",
    "neutral",
    "negative",
    "mixed",
    "nostalgic",
    "melancholic",
    "aggressive",
    "horny",
    "hopeful",
    "dramatic"
]

STYLES = [
    "old_vk_wall_post",
    "messy_emotional",
    "capslock_chaos",
    "internet_slang",
    "poetic",
    "dramatic_prose",
    "normal_conversational",
    "drunk_post",
    "shitpost",
    "repost_commentary",
    "late_night_post",
    "music_status_core"
]

STRUCTURES = [
    "very_short",
    "single_sentence",
    "multi_sentence",
    "long_paragraph",
    "poem_like",
    "dialogue_like",
    "fragmented_stream"
]

ARCHETYPES = [
    "sad_night_post",
    "music_obsession_post",
    "relationship_wall_post",
    "drunk_existential_post",
    "aggressive_comment_war",
    "repost_with_commentary",
    "concert_hype",
    "random_life_fragment",
    "edgy_teen_post",
    "nostalgic_memory_post",
    "capslock_emotional_breakdown",
    "horny_flirty_post"
]


PROMPT = """
You are auditing a latent taxonomy for Russian VK-style social media texts.

Your task:
1. Classify the text using the EXISTING taxonomy.
2. Decide whether the taxonomy sufficiently covers the text.
3. If not, propose a NEW category.

Text:
\"\"\"{text}\"\"\"

Existing taxonomy:

TOPICS:
{topics}

INTENTS:
{intents}

EMOTIONS:
{emotions}

STYLES:
{styles}

STRUCTURES:
{structures}

ARCHETYPES:
{archetypes}

Return strictly JSON:

{{
  "topic": "...",
  "intent": "...",
  "emotion": "...",
  "style": "...",
  "structure": "...",
  "archetype": "...",

  "taxonomy_fit": true,

  "fit_score": 1,

  "missing_category_type": null,

  "proposed_new_category": null,

  "reasoning": "short explanation"
}}

Rules:
- fit_score: integer from 1 to 5
- fit_score=5 means taxonomy perfectly fits
- fit_score=1 means taxonomy fails badly
- taxonomy_fit=false only if current taxonomy is clearly insufficient
- missing_category_type must be one of:
  "topic", "style", "emotion", "structure", "archetype", "intent"
- proposed_new_category must be concise snake_case
"""


client = AsyncOpenAI()

CONCURRENCY = 20
MODEL = "gpt-5.4-mini-2026-03-17"

semaphore = asyncio.Semaphore(CONCURRENCY)


async def classify_text(text):

    prompt = PROMPT.format(
        text=text,
        topics=", ".join(TOPICS),
        intents=", ".join(INTENTS),
        emotions=", ".join(EMOTIONS),
        styles=", ".join(STYLES),
        structures=", ".join(STRUCTURES),
        archetypes=", ".join(ARCHETYPES)
    )

    async with semaphore:

        response = await client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.2,
            max_completion_tokens=300
        )

    content = response.choices[0].message.content.strip()

    if content.startswith("```"):
        content = content.split("```")[1]

    return json.loads(content)

In [19]:
df = pd.read_csv("../data/raw/rusentiment_preselected_posts.csv")

df = df[df["label"] != "skip"]

df = df.sample(
    n=500,
    random_state=42
).reset_index(drop=True)

tasks = [
    classify_text(text)
    for text in df["text"]
]

results = []

for coro in tqdm(
    asyncio.as_completed(tasks),
    total=len(tasks)
):
    try:
        results.append(await coro)

    except Exception as e:
        results.append({
            "error": str(e)
        })

result_df = pd.concat(
    [
        df,
        pd.DataFrame(results)
    ],
    axis=1
)

100%|██████████| 500/500 [01:30<00:00,  5.53it/s]


In [20]:
print("\nTOPIC DISTRIBUTION")
print(
    result_df["topic"]
    .value_counts(normalize=True)
)

print("\nSTYLE DISTRIBUTION")
print(
    result_df["style"]
    .value_counts(normalize=True)
)

print("\nFIT SCORE DISTRIBUTION")
print(
    result_df["fit_score"]
    .value_counts(normalize=True)
    .sort_index()
)

print("\nLOW FIT EXAMPLES")
low_fit = result_df[
    result_df["fit_score"] <= 2
]

print(
    low_fit[
        [
            "text",
            "proposed_new_category",
            "missing_category_type",
            "reasoning"
        ]
    ].head(20)
)

print("\nPROPOSED NEW CATEGORIES")
print(
    result_df["proposed_new_category"]
    .dropna()
    .value_counts()
    .head(30)
)

result_df.to_csv(
    "../data/taxonomy_audit.csv",
    index=False
)


TOPIC DISTRIBUTION
topic
relationships     0.198
short_reaction    0.172
daily_life        0.150
toxicity          0.094
philosophical     0.084
humor             0.084
other             0.068
discussion        0.056
creative          0.056
question          0.026
storytelling      0.010
music             0.002
Name: proportion, dtype: float64

STYLE DISTRIBUTION
style
normal_conversational    0.416
internet_slang           0.210
poetic                   0.130
messy_emotional          0.088
capslock_chaos           0.046
old_vk_wall_post         0.030
music_status_core        0.022
shitpost                 0.022
repost_commentary        0.018
dramatic_prose           0.012
drunk_post               0.006
Name: proportion, dtype: float64

FIT SCORE DISTRIBUTION
fit_score
3    0.024
4    0.702
5    0.274
Name: proportion, dtype: float64

LOW FIT EXAMPLES
Empty DataFrame
Columns: [text, proposed_new_category, missing_category_type, reasoning]
Index: []

PROPOSED NEW CATEGORIES
Series([], 

In [21]:
SURFACE_PROMPT = """
You are analyzing realism features of Russian VK-style social media texts.

Text:
\"\"\"{text}\"\"\"

Analyze the following surface features:

1. capslock
- none
- light
- heavy

2. emoji_usage
- none
- light
- heavy

3. typos
- clean
- light_typos
- heavy_typos

4. profanity
- none
- light
- heavy

5. punctuation
- normal
- repeated_exclamations
- ellipsis_heavy
- chaotic

6. internet_markers
Examples:
:), :D, xD, ))), ахах, бля, еее, ору, лол

Return strictly JSON:

{{
  "capslock": "...",
  "emoji_usage": "...",
  "typos": "...",
  "profanity": "...",
  "punctuation": "...",
  "internet_markers_present": true
}}
"""


async def classify_surface_features(text):

    prompt = SURFACE_PROMPT.format(
        text=text
    )

    async with semaphore:

        response = await client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0.1,
            max_completion_tokens=150
        )

    content = response.choices[0].message.content.strip()

    if content.startswith("```"):
        content = content.split("```")[1]

    return json.loads(content)


surface_tasks = [
    classify_surface_features(text)
    for text in df["text"]
]

surface_results = []

for coro in tqdm(
    asyncio.as_completed(surface_tasks),
    total=len(surface_tasks)
):
    try:
        surface_results.append(await coro)

    except Exception as e:
        surface_results.append({
            "error": str(e)
        })

surface_df = pd.concat(
    [
        df.reset_index(drop=True),
        pd.DataFrame(surface_results)
    ],
    axis=1
)

100%|██████████| 500/500 [00:34<00:00, 14.44it/s]


In [22]:
print("\nCAPSLOCK")
print(
    surface_df["capslock"]
    .value_counts(normalize=True)
)

print("\nEMOJI USAGE")
print(
    surface_df["emoji_usage"]
    .value_counts(normalize=True)
)

print("\nTYPOS")
print(
    surface_df["typos"]
    .value_counts(normalize=True)
)

print("\nPROFANITY")
print(
    surface_df["profanity"]
    .value_counts(normalize=True)
)

print("\nPUNCTUATION")
print(
    surface_df["punctuation"]
    .value_counts(normalize=True)
)

print("\nINTERNET MARKERS")
print(
    surface_df["internet_markers_present"]
    .value_counts(normalize=True)
)

print("\nPROFANITY x LABEL")
print(
    pd.crosstab(
        surface_df["profanity"],
        surface_df["label"],
        normalize="columns"
    )
)

print("\nEMOJI x LABEL")
print(
    pd.crosstab(
        surface_df["emoji_usage"],
        surface_df["label"],
        normalize="columns"
    )
)

surface_df.to_csv(
    "../data/surface_feature_audit.csv",
    index=False
)


CAPSLOCK
capslock
none     0.924
light    0.048
heavy    0.028
Name: proportion, dtype: float64

EMOJI USAGE
emoji_usage
none     0.872
light    0.120
heavy    0.008
Name: proportion, dtype: float64

TYPOS
typos
clean          0.734
light_typos    0.254
heavy_typos    0.012
Name: proportion, dtype: float64

PROFANITY
profanity
none     0.842
light    0.092
heavy    0.066
Name: proportion, dtype: float64

PUNCTUATION
punctuation
normal                   0.660
ellipsis_heavy           0.216
repeated_exclamations    0.102
chaotic                  0.022
Name: proportion, dtype: float64

INTERNET MARKERS
internet_markers_present
False    0.76
True     0.24
Name: proportion, dtype: float64

PROFANITY x LABEL
label      negative   neutral  positive  speech
profanity                                      
heavy      0.050847  0.069106  0.077586    0.05
light      0.076271  0.093496  0.103448    0.10
none       0.872881  0.837398  0.818966    0.85

EMOJI x LABEL
label        negative   neutral 